In [ ]:
# Example 1: Construct a Celery app and inspect basic identity attributes.
from celery import Celery

app = Celery('smoke_test', broker='memory://', backend='cache+memory://')

print(f"app.main      = {app.main}")
print(f"app.conf.broker_url = {app.conf.broker_url}")
print(f"app.conf.result_backend = {app.conf.result_backend}")
print(f"type(app)     = {type(app).__module__}.{type(app).__name__}")

assert app.main == 'smoke_test'
assert isinstance(app, Celery)


In [ ]:
# Example 2: Register a task via the @app.task decorator and verify task registry.
from celery import Celery

app = Celery('reg_test', broker='memory://', backend='cache+memory://')

@app.task
def add(x, y):
    return x + y

@app.task(name='custom.multiply')
def multiply(x, y):
    return x * y

print(f"add.name = {add.name}")
print(f"multiply.name = {multiply.name}")
print(f"registered tasks include add: {add.name in app.tasks}")
print(f"registered tasks include custom.multiply: {'custom.multiply' in app.tasks}")

# Direct invocation (bypasses broker) still works for unit tests.
assert add(2, 3) == 5
assert multiply(4, 6) == 24
assert 'custom.multiply' in app.tasks


In [ ]:
# Example 3: Use task signatures and chain/group/chord canvas primitives (construction only).
from celery import Celery, chain, group, chord

app = Celery('canvas_test', broker='memory://', backend='cache+memory://')

@app.task
def add(x, y):
    return x + y

sig = add.s(2, 3)
print(f"signature: {sig}")
print(f"signature args: {sig.args}")

c = chain(add.s(1, 2), add.s(3))
g = group(add.s(i, i) for i in range(3))
ch = chord([add.s(1, 1), add.s(2, 2)], add.s(10))

print(f"chain tasks: {len(c.tasks)}")
print(f"group tasks: {len(g.tasks)}")
print(f"chord header: {len(ch.tasks)}")

assert sig.args == (2, 3)
assert len(c.tasks) == 2


In [ ]:
# Example 4: Configure app via conf.update + verify config round-trip.
from celery import Celery

app = Celery('conf_test')
app.conf.update(
    broker_url='memory://',
    result_backend='cache+memory://',
    task_serializer='json',
    accept_content=['json'],
    result_serializer='json',
    timezone='UTC',
    enable_utc=True,
    task_default_queue='smoke',
)

print(f"broker_url      = {app.conf.broker_url}")
print(f"task_serializer = {app.conf.task_serializer}")
print(f"timezone        = {app.conf.timezone}")
print(f"default_queue   = {app.conf.task_default_queue}")

assert app.conf.task_serializer == 'json'
assert app.conf.enable_utc is True
assert app.conf.task_default_queue == 'smoke'


In [ ]:
# Example 5: Periodic schedule construction with celery.schedules.
from celery import Celery
from celery.schedules import crontab, schedule
from datetime import timedelta

app = Celery('beat_test', broker='memory://', backend='cache+memory://')

@app.task
def heartbeat():
    return 'ok'

app.conf.beat_schedule = {
    'every-30-seconds': {
        'task': heartbeat.name,
        'schedule': timedelta(seconds=30),
    },
    'every-midnight': {
        'task': heartbeat.name,
        'schedule': crontab(hour=0, minute=0),
    },
}

cron = crontab(hour=0, minute=0)
sched = schedule(timedelta(seconds=30))

print(f"crontab repr:  {cron}")
print(f"schedule repr: {sched}")
print(f"beat entries:  {list(app.conf.beat_schedule.keys())}")

assert 'every-30-seconds' in app.conf.beat_schedule
assert 'every-midnight' in app.conf.beat_schedule
assert cron.hour == {0}
